# 图像特征提取 — CLIP ViT-L/14 (768-dim)

## 嵌入向量模型对比：ViT-B/32 / ViT-L/14

| | ViT-B/32 | ViT-L/14 | 提升 |
|------|:---:|:---:|------|
| 模型大小 | 581 MB | 1.71 GB | 3× |
| 输出维度 | 512 | 768 | 1.5× |
| Patch 大小 | 32×32 | 14×14 | 更细粒度 |
| 层数 | 12 | 24 | 更深的语义层次 |
| ImageNet Top-1 | 63.4% | 75.5% | +12% |

关键差异：
- **Patch 14 比 Patch 32**：14×14 的 patch 意味着每张图切成更多小块（vs 32×32 的粗粒度），能捕获更细的视觉细节——对电影海报的文字、图书封面的设计风格等细粒度特征
- **24 层比 12 层**：更深的 Transformer 能建模更复杂的视觉语义层次，有助于区分"同一系列的不同续集封面"这种细微差异
- **768d 比 512d**：更大的嵌入空间提供更强的区分度，代价是多 ~0.5M 模型参数（下游投影层 768→256 vs 512→256）

### 与 ViT-B/32 对比

| 指标 | ViT-B/32 | ViT-L/14 |
|------|:---:|:---:|
| 域内 cosine mean | 0.48 | 0.52 |
| 域间 cosine mean | 0.39 | 0.42 |
| 域分离度 (Δ cosine) | 0.09 | 0.10 |
| 零向量 | 19 | 19 |

ViT-L/14 的域内 cosine 更高，说明同域物品的视觉表示更紧凑；域分离度略好于 B/32。

In [ ]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch, json, numpy as np, pandas as pd
from PIL import Image
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import CLIPProcessor, CLIPModel

META_CSV    = "final/item_meta_merged.csv"
IMAGE_MAP   = "final/item_image_map.json"
OUTPUT_DIR  = "final/features"
CLIP_MODEL  = "openai/clip-vit-large-patch14"
BATCH_SIZE  = 32
FEATURE_DIM = 768       # ViT-L/14 输出维度

os.makedirs(OUTPUT_DIR, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

Device: cuda


### 模型加载

In [ ]:
model = CLIPModel.from_pretrained(CLIP_MODEL).to(device).eval()
processor = CLIPProcessor.from_pretrained(CLIP_MODEL)
torch.set_grad_enabled(False)
print(f"Loaded: {CLIP_MODEL}")

d:\Python2\Python311\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   1%|          | 10.5M/1.71G [00:00<?, ?B/s]

d:\Python2\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--openai--clip-vit-large-patch14. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


preprocessor_config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Loaded: openai/clip-vit-large-patch14


In [ ]:
meta = pd.read_csv(META_CSV)
print(f"Items: {len(meta)}")

with open(IMAGE_MAP) as f:
    img_map = json.load(f)

img_paths = {}
missing = 0
for _, row in meta.iterrows():
    iid = int(row["parent_asin"])
    key = str(iid)
    if key in img_map:
        path = img_map[key].replace("\\", "/")
        if not path.startswith("final/"):
            path = f"final/images/{iid}.jpg"
    else:
        path = None
        missing += 1
    img_paths[iid] = path

print(f"Mapped: {len(img_paths)-missing}  Missing: {missing} (zero-fill)")

Items: 43528
Mapped: 43509  Missing: 19 (zero-fill)


###  Dataset 设计

In [ ]:
class ImageDataset(Dataset):
    def __init__(self, item_ids, path_dict):
        self.ids = list(item_ids)
        self.paths = [path_dict[i] for i in self.ids]
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx): return self.ids[idx], self.paths[idx]

def collate(batch):
    return [b[0] for b in batch], [b[1] for b in batch]

item_ids = sorted(img_paths.keys())
loader = DataLoader(ImageDataset(item_ids, img_paths),
                    batch_size=BATCH_SIZE, shuffle=False,
                    collate_fn=collate, num_workers=0)
print(f"Batches: {len(loader)} (x{BATCH_SIZE} = ~{len(loader)*BATCH_SIZE} imgs)")

Batches: 1361 (x32 = ~43552 imgs)


###  特征提取流水线

核心逻辑（逐 batch）：

1. **加载图片**：`Image.open(p).convert("RGB")`
2. **CLIP 预处理**：`processor(images=...)` 完成 resize→center crop→normalize
   - ViT-L/14 输入：224×224
3. **提取特征**：`model.get_image_features()` 输出 (B, 768) 向量
4. **缺失填充**：加载失败的样本填 `np.zeros(768)`，保持行索引不变

In [ ]:
all_feats, all_ids, fail_count = [], [], 0

for ids_batch, paths_batch in tqdm(loader, desc="ViT-L/14"):
    images, valid_ix = [], []
    for i, p in enumerate(paths_batch):
        # 图片加载
        if p and os.path.exists(p):
            try:
                images.append(Image.open(p).convert("RGB"))
                valid_ix.append(i)
            except Exception:
                pass
    # 特征提取
    feats = np.zeros((len(ids_batch), FEATURE_DIM), dtype=np.float32)
    if images:
        # CLIP 预处理 + 前向传播
        inp = processor(images=images, return_tensors="pt").to(device)
        vf = model.get_image_features(**inp).cpu().numpy()
        for j, f in zip(valid_ix, vf):
            feats[j] = f
    else:
        fail_count += len(ids_batch)

    all_feats.append(feats)
    all_ids.extend(ids_batch)

all_feats = np.concatenate(all_feats, axis=0)
print(f"Done. shape={all_feats.shape}  zero_rows={(all_feats.sum(1)==0).sum()} (expect ~19)")

ViT-L/14: 100%|██████████| 1361/1361 [57:23<00:00,  2.53s/it] 


Done. shape=(43528, 768)  zero_rows=19 (expect ~19)


In [ ]:
np.save(f"{OUTPUT_DIR}/image_features_768.npy", all_feats)
pd.DataFrame({"item_id": all_ids}).to_csv(f"{OUTPUT_DIR}/image_id_map_vitl.csv", index=False)
print(f"Saved: {OUTPUT_DIR}/image_features_768.npy")
print(f"Saved: {OUTPUT_DIR}/image_id_map_vitl.csv")

Saved: final/features/image_features_768.npy
Saved: final/features/image_id_map_vitl.csv
